# Wireframe de mano con Intel RealSense Depth Camera + MediaPipe

Detecta **una mano derecha** en tiempo real y dibuja su esqueleto (21 landmarks y conexiones) sobre el video de color y profundidad de la cámara RealSense.

**Flujo**
1. RealSense entrega frames de color (640x480 @ 30 fps, BGR).
2. Se alinean espacialmente las imagenes de color y profundidad con `align = rs.align(rs.stream.color)`
3. Se convierte a RGB (MediaPipe lo requiere).
4. `mp.solutions.hands` extrae los 21 landmarks y la lateralidad (`Left`/`Right`).
5. Si la mano es `Right`, se dibuja el wireframe y el texto *Mano Derecha Detectada*.
6. Se muestra en una ventana OpenCV. Tecla **`q`** para salir.

**Notas**
- En esta versión se implementa la detección de profundidad de la camara RealSense, en donde se dibuja el Wireframe sobre **ambas imágenes**.
- MediaPipe asume imagen tipo espejo (selfie). Como la RealSense no refleja la imagen, la etiqueta `Right` puede corresponder a la mano **izquierda** real. Como sucede esto, se cambia el filtro a `"Left"`.

## 1. Imports y configuración de MediaPipe

In [1]:
import warnings
import pyrealsense2 as rs
import numpy as np
import cv2
import mediapipe as mp

warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf.symbol_database")
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,          # Buscar máximo 1 mano en pantalla
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## 2. Captura RealSense y bucle de detección
Ejecutar la celda; presionar `q` sobre la ventana de video para terminar.

In [2]:
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)

align_to = rs.stream.color
align = rs.align(align_to)

print("Iniciando cámara... Presiona la tecla 'q' sobre la ventana de video para salir.")
pipeline.start(config)

try:
    while True:
        # Esperar fotogramas crudos
        frames = pipeline.wait_for_frames()
        
        # Procesar la alineación espacial antes de extraerlos
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not depth_frame:
            continue

        color_image = np.asanyarray(color_frame.get_data())
        depth_image = np.asanyarray(depth_frame.get_data())

        # Convertir la matriz de profundidad (16 bits) a mapa de color térmico (8 bits) para visualizarla
        depth_colormap = cv2.applyColorMap(cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET)

        # Conversion de BGR a RGB para MediaPipe
        image_rgb = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)

        # Extraer los 21 puntos clave
        results = hands.process(image_rgb)

        # Dibujar el wireframe si se detecta una mano
        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):

                # Filtrar mano derecha
                label = handedness.classification[0].label
                if label == "Left":
                    # Trazar el esqueleto sobre la imagen a color original
                    mp_drawing.draw_landmarks(
                        color_image,
                        hand_landmarks,
                        mp_hands.HAND_CONNECTIONS
                    )
                    # Trazar exactamente el mismo esqueleto sobre el mapa de profundidad
                    mp_drawing.draw_landmarks(
                        depth_colormap,
                        hand_landmarks,
                        mp_hands.HAND_CONNECTIONS
                    )
                    
                    cv2.putText(color_image, "Color", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    cv2.putText(depth_colormap, "Profundidad", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # Apilar y mostrar ambas imágenes horizontalmente en un solo lienzo
        images_hstack = np.hstack((color_image, depth_colormap))
        cv2.imshow('Wireframe Dual - Intel RealSense', images_hstack)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    print("Pipeline detenido y ventanas cerradas.")

W0000 00:00:1792524138.403200   17237 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792524138.427426   17237 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Iniciando cámara... Presiona la tecla 'q' sobre la ventana de video para salir.
Pipeline detenido y ventanas cerradas.


## Solución de problemas

| Error | Causa / solución |
|---|---|
| `RuntimeError: No device connected` | RealSense no conectada o sin permisos (udev). |
| Kernel muere al importar | Libs nativas incompatibles o poca RAM; probar el script en terminal. |